# Split Train and Validation Dataset

This notebook creates a reproducible image-level train/validation split from the final merged CXRaide training annotation file: `01_merged_nih_vinbig_training_boxes.csv`.

The source CSV uses this schema:

`source,image_id,label_name,x_min,y_min,x_max,y_max`

This notebook adds two training-friendly columns:

- `target_class_id`: stable numeric class ID generated from `label_name`
- `split`: `train` or `val`

The split is done by `source + image_id`, not annotation row, so all boxes from the same image stay in one split.

## Setup

Import path helpers, random utilities, and pandas for CSV processing.

In [11]:
# Notebook guide: shared imports for creating a reproducible image-level split.
# Pandas handles annotation tables; `random` controls deterministic image selection.

from pathlib import Path
import random

import pandas as pd

## Configuration

Define the source annotation CSV, output CSV names, validation size, random seed, and rare classes to monitor during the split.

In [12]:
# Notebook guide: configure the split source and outputs.
# All output files start with `04_` to match this preparation step.

PROCESSED_DIR = Path("../data/processed")
SOURCE_CSV = PROCESSED_DIR / "01_merged_nih_vinbig_training_boxes.csv"

SPLIT_CSV = PROCESSED_DIR / "04_merged_train_val_split_1024.csv"
TRAIN_CSV = PROCESSED_DIR / "04_merged_train_split_1024.csv"
VAL_CSV = PROCESSED_DIR / "04_merged_val_split_1024.csv"

VAL_FRACTION = 0.15
SEED = 42
RARE_CLASS_NAMES = {"Consolidation", "Atelectasis", "Pneumothorax"}

# Stable class order for the detector. Keep this fixed across future runs.
CLASS_NAME_TO_ID = {
    "Cardiomegaly": 1,
    "Pleural thickening": 2,
    "Pulmonary fibrosis": 3,
    "Pleural effusion": 4,
    "Nodule/Mass": 5,
    "Infiltration": 6,
    "Consolidation": 7,
    "Atelectasis": 8,
    "Pneumothorax": 9,
}

## Load Final Merged Annotations

Read the final merged annotation CSV and verify the expected detection columns are available before creating splits.

In [13]:
# Notebook guide: load the final merged CXRaide annotations and validate the schema.
# This is the source of truth after NIH and VinBig have been merged into one training table.

df = pd.read_csv(SOURCE_CSV)
required_columns = {"source", "image_id", "label_name", "x_min", "y_min", "x_max", "y_max"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df["source"] = df["source"].astype(str)
df["image_id"] = df["image_id"].astype(str)
df["label_name"] = df["label_name"].astype(str)

unknown_labels = sorted(set(df["label_name"]) - set(CLASS_NAME_TO_ID))
if unknown_labels:
    raise ValueError(f"Labels missing from CLASS_NAME_TO_ID: {unknown_labels}")

df["target_class_id"] = df["label_name"].map(CLASS_NAME_TO_ID).astype(int)

print(f"Annotation rows: {len(df):,}")
print(f"Unique source/image pairs: {df[['source', 'image_id']].drop_duplicates().shape[0]:,}")
display(df.groupby(["source", "label_name"]).size().sort_values())

Annotation rows: 15,480
Unique source/image pairs: 4,496


source  label_name        
NIH     Infiltration            42
        Pneumothorax            63
        Cardiomegaly            78
        Pleural effusion        81
        Atelectasis            100
VinBig  Pneumothorax           132
        Atelectasis            231
        Consolidation          434
        Infiltration           960
        Pleural effusion      1774
        Nodule/Mass           1865
        Cardiomegaly          2300
        Pulmonary fibrosis    3364
        Pleural thickening    4056
dtype: int64

## Create Class-Aware Image Split

Select validation images using class deficits. At each step, the next validation image is the one that best fills the current per-class validation target. Rare classes receive extra priority so Consolidation, Atelectasis, and Pneumothorax are represented in validation.

In [14]:
# Notebook guide: create an image-level split that tries to preserve class coverage.
# The image key includes `source` so identical NIH/VinBig IDs cannot collide.

rng = random.Random(SEED)
split_df = df.copy()
split_df["image_key"] = split_df["source"] + "::" + split_df["image_id"]

image_label_counts = (
    split_df.groupby(["image_key", "label_name"])
    .size()
    .unstack(fill_value=0)
)
image_keys = list(image_label_counts.index)
target_val_images = max(1, round(len(image_keys) * VAL_FRACTION))

class_counts = split_df.groupby("label_name").size()
target_val_counts = (class_counts * VAL_FRACTION).round().clip(lower=1).astype(int)
val_counts = pd.Series(0, index=class_counts.index, dtype=float)
remaining = set(image_keys)
val_keys = set()

rare_boost = pd.Series(1.0, index=class_counts.index)
rare_boost.loc[rare_boost.index.intersection(RARE_CLASS_NAMES)] = 2.0

while remaining and len(val_keys) < target_val_images:
    deficits = (target_val_counts - val_counts).clip(lower=0)
    normalized_deficits = (deficits / target_val_counts).fillna(0) * rare_boost
    candidate_scores = image_label_counts.loc[list(remaining)].clip(upper=deficits, axis=1).mul(normalized_deficits, axis=1).sum(axis=1)
    candidate_scores = candidate_scores + pd.Series(
        {image_key: rng.random() * 1e-6 for image_key in candidate_scores.index}
    )
    selected = candidate_scores.idxmax()
    val_keys.add(selected)
    remaining.remove(selected)
    val_counts = val_counts.add(image_label_counts.loc[selected], fill_value=0)

train_keys = set(image_keys) - val_keys
split_df["split"] = split_df["image_key"].map(lambda image_key: "val" if image_key in val_keys else "train")
split_df = split_df.drop(columns=["image_key"])

print(f"Train images: {len(train_keys):,}")
print(f"Val images:   {len(val_keys):,}")
print(f"Val image fraction: {len(val_keys) / len(image_keys):.3f}")

Train images: 3,822
Val images:   674
Val image fraction: 0.150


## Verify Split Distribution

Compare annotation counts per class in train and validation. The validation split should include every rare class so model evaluation can track them.

In [15]:
# Notebook guide: inspect class counts before writing files.
# This check helps catch rare classes that accidentally disappeared from validation.

summary = (
    split_df.groupby(["label_name", "split"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda table: table.sum(axis=1))
)
summary["val_fraction"] = (summary.get("val", 0) / summary["total"]).round(3)
display(summary.sort_values("total"))

rare_summary = summary.loc[summary.index.intersection(RARE_CLASS_NAMES)]
if (rare_summary.get("val", 0) == 0).any():
    missing = rare_summary[rare_summary.get("val", 0) == 0].index.tolist()
    raise ValueError(f"Rare classes missing from validation split: {missing}")

split,train,val,total,val_fraction
label_name,,,,
Pneumothorax,150,45,195,0.231
Atelectasis,260,71,331,0.215
Consolidation,339,95,434,0.219
Infiltration,783,219,1002,0.219
Pleural effusion,1410,445,1855,0.240
Nodule/Mass,1378,487,1865,0.261
Cardiomegaly,1896,482,2378,0.203
Pulmonary fibrosis,2632,732,3364,0.218
Pleural thickening,3106,950,4056,0.234


## Write Split CSVs

Write one combined split-aware CSV plus separate train and validation CSVs. The training notebook reads the combined file by default.

In [16]:
# Notebook guide: save the split outputs under `data/processed` with the `04_` prefix.
# The combined CSV is the source of truth; separate train/val CSVs are convenience exports.

train_df = split_df[split_df["split"] == "train"].copy()
val_df = split_df[split_df["split"] == "val"].copy()

split_df.to_csv(SPLIT_CSV, index=False)
train_df.to_csv(TRAIN_CSV, index=False)
val_df.to_csv(VAL_CSV, index=False)

print(f"Wrote combined split CSV: {SPLIT_CSV}")
print(f"Wrote train CSV:          {TRAIN_CSV}")
print(f"Wrote val CSV:            {VAL_CSV}")

Wrote combined split CSV: ..\data\processed\04_merged_train_val_split_1024.csv
Wrote train CSV:          ..\data\processed\04_merged_train_split_1024.csv
Wrote val CSV:            ..\data\processed\04_merged_val_split_1024.csv
